### Import dependencies

In [11]:
import openai
from qdrant_client import QdrantClient

### Embedding funtion

In [8]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval functions

In [13]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [5]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    return results

In [24]:
retrieved_context = retrieve_data("Do you have any Manual Food Processor Vegetable Chopper?", k=10)

In [25]:
retrieved_context

QueryResponse(points=[ScoredPoint(id=3, version=1, score=0.63702655, payload={'preprocessed_description': "Manual Food Processor Vegetable Chopper, Geedel Pull Chopper with String, Food Chopper Vegetable Cutter for Veggies, Fruits, Salad, Onion, Ginger, Nuts, Herbs, etc, 2 Cup(500ml), Grey 🍅Chopping in A Breeze - The 3 blades of the vegetable chopper are distributed at different heights and directions for more efficient and evenly cutting, chopping in seconds. Just get the kitchen job done with several pulls, pulling the rope once equals at least 20 cuts with a knife. 1-5 times for roughly food in 5s, 6-10 for medium in 10s, 11-20 for fine and 20+ for extra fine and liquid purees. What's more, the scraper sweeps the ingredients off the lid while cutting, to help chop fully. 🥕 Features Your All Need - The chopper vetetable cutter with 2 cups is enough for your kitchen needs, also suitable for small amount of seasoniong. Meet your different needs including veggie such as onion, garlic, c

In [14]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieve_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context.append(result.payload['preprocessed_description'])
        similarity_scores.append(result.score)
        retrieve_context_ratings.append(result.payload['average_rating'])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieve_context_ratings": retrieve_context_ratings
    }

In [15]:
retrieved_context = retrieve_data("Do you have any Manual Food Processor Vegetable Chopper?", k=10)

In [16]:
retrieved_context

{'retrieved_context_ids': ['B08LZCRF9X',
  'B0B2RKLYK3',
  'B0B7PB86G7',
  'B09XQ64DMH',
  'B07CZ1L3RW',
  'B09TPR27NX',
  'B0BKQTRCC3',
  'B0CD3L7YKM',
  'B0BPB9JBH7',
  'B0BT99ZKD8'],
 'retrieved_context': ["Manual Food Processor Vegetable Chopper, Geedel Pull Chopper with String, Food Chopper Vegetable Cutter for Veggies, Fruits, Salad, Onion, Ginger, Nuts, Herbs, etc, 2 Cup(500ml), Grey 🍅Chopping in A Breeze - The 3 blades of the vegetable chopper are distributed at different heights and directions for more efficient and evenly cutting, chopping in seconds. Just get the kitchen job done with several pulls, pulling the rope once equals at least 20 cuts with a knife. 1-5 times for roughly food in 5s, 6-10 for medium in 10s, 11-20 for fine and 20+ for extra fine and liquid purees. What's more, the scraper sweeps the ingredients off the lid while cutting, to help chop fully. 🥕 Features Your All Need - The chopper vetetable cutter with 2 cups is enough for your kitchen needs, also suita

### Format Retrieved Context

In [21]:
def proccess_context(context):
    formatted_context = ''
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieve_context_ratings']):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [22]:
processed_context = proccess_context(retrieved_context)

In [23]:
print(processed_context)

- ID: B08LZCRF9X, rating: 4.4, description: Manual Food Processor Vegetable Chopper, Geedel Pull Chopper with String, Food Chopper Vegetable Cutter for Veggies, Fruits, Salad, Onion, Ginger, Nuts, Herbs, etc, 2 Cup(500ml), Grey 🍅Chopping in A Breeze - The 3 blades of the vegetable chopper are distributed at different heights and directions for more efficient and evenly cutting, chopping in seconds. Just get the kitchen job done with several pulls, pulling the rope once equals at least 20 cuts with a knife. 1-5 times for roughly food in 5s, 6-10 for medium in 10s, 11-20 for fine and 20+ for extra fine and liquid purees. What's more, the scraper sweeps the ingredients off the lid while cutting, to help chop fully. 🥕 Features Your All Need - The chopper vetetable cutter with 2 cups is enough for your kitchen needs, also suitable for small amount of seasoniong. Meet your different needs including veggie such as onion, garlic, carrot, tomato, etc; fruits such as avocado, apple, etc; and oth

### Create prompt template function

In [24]:
def build_prompt(query, preprocessed_context):
    prompt = f"""
            You are a shopping assistant that can answer questions about the products in stock.

            You will be given a question and a list of context.

            Instructions:
            - Answer the question based on the provided context only.
            - Never use word context and refer to it as the available products.
            - Do not use markdown formatting.

            Context:
            {preprocessed_context}

            Question:
            {query}    
            """
    return prompt


In [25]:
prompt = build_prompt("Do you have any Manual Food Processor Vegetable Chopper?", processed_context)

In [26]:
print(prompt)


            You are a shopping assistant that can answer questions about the products in stock.

            You will be given a question and a list of context.

            Instructions:
            - Answer the question based on the provided context only.
            - Never use word context and refer to it as the available products.
            - Do not use markdown formatting.

            Context:
            - ID: B08LZCRF9X, rating: 4.4, description: Manual Food Processor Vegetable Chopper, Geedel Pull Chopper with String, Food Chopper Vegetable Cutter for Veggies, Fruits, Salad, Onion, Ginger, Nuts, Herbs, etc, 2 Cup(500ml), Grey 🍅Chopping in A Breeze - The 3 blades of the vegetable chopper are distributed at different heights and directions for more efficient and evenly cutting, chopping in seconds. Just get the kitchen job done with several pulls, pulling the rope once equals at least 20 cuts with a knife. 1-5 times for roughly food in 5s, 6-10 for medium in 10s, 11-20 for f

### Generate answer function

In [27]:
def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="none"
    )
    return response.choices[0].message.content

### Combine RAG pipeline

In [28]:
def rag_pipeline(query, top_k=5):
    retrieved_context = retrieve_data(query, top_k)
    processed_context = proccess_context(retrieved_context)
    prompt = build_prompt(query, processed_context)
    answer = generate_answer(prompt)
    return answer

In [29]:
print(rag_pipeline("Do you have any Manual Food Processor Vegetable Chopper?"))

Yes. We have the Geedel manual food processor vegetable chopper (pull chopper) with a 2 cup (500ml) capacity, model ID B08LZCRF9X. It’s designed for chopping vegetables, fruits, onions, ginger, nuts, herbs, and more.


In [30]:
print(rag_pipeline("Could you suggest some humidifiers for my room?"))

Yes—here are some humidifier options available:

1) GANIZA 50H Air Vaporizer Humidifier (ID: B0B7XNXFXP, rating 4.5)
- Type/coverage: Cool mist humidifier for large rooms up to 430 ft²
- Tank: 4L (about 1 gallon), runs up to 50 hours
- Mist control: Adjustable output up to 300 ml/h with a 360° rotating nozzle
- Noise: Quiet (below 22 dB) for bedrooms, nurseries, offices, and plant grow rooms
- Extra features: Filter-free and designed to prevent wet surfaces/leaks; bottom-fill with visible water line and automatic water shortage protection

If you tell me the approximate size of your room (in ft²) and whether it’s for a bedroom/kids/plants, I can help you pick the best fit from what’s available.


In [31]:
print(rag_pipeline("Could you suggest some bed products? I'm looking for a good rating on those"))

Sure. Here are some bed-related products from the available products with good ratings:

1) Feonase King Size Bed Frame (Modern Velvet Upholstered Platform Bed Frame) — rating 4.6
2) Rosevera Clancy Adjustable Height Headboard (King, Linen Gray) — rating 4.5
3) Yaheetech Full Bed Frame Metal Platform Bed with Under-Bed Storage (Antique Gold) — rating 4.7
4) Full XL Pure Egyptian Cotton Bed Sheets Set (4 Pc, 1000 Thread Count, Aqua Blue) — rating 4.4
5) Side Sleeper Cube Memory Foam Pillow (15×12×6inch) — rating 3.8

If you tell me which size you need (king, full, or full-XL) and whether you want a frame, headboard, sheets, or a pillow, I can narrow it down.
